# Share the Research and Attack the Analysis

**Topic 09 · 2 lectures**

<hr>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb09_attack_the_analysis_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** all positions (attacking your own analysis). A finding can
sit at any compass position, descriptive or causal, at the data at hand or a
population or unseen cases. Whatever position yours takes, this week you try to
break it on purpose before anyone else can. The kind of attack changes with the
position, but the discipline does not.

**Design pathway:** cross-cutting (robustness). This is not a new design library
entry. It is the toolkit that stress-tests whatever design you already declared,
so it rides alongside every pathway from Weeks 5 through 9.

| | |
|---|---|
| **A claim this topic PERMITS** | "My claim survives these pre-listed robustness and placebo checks; where it moves, I report by how much and why." |
| **A claim this topic does NOT permit** | "The result is significant" reported from the one specification that worked, after quietly trying many, with the search never disclosed. That is specification searching. |

**Where this sits in the course:** Week 10, the attack-the-analysis unit. It takes
the estimate you produced in M8 and stress-tests it, then feeds M9, your poster
draft plus research audit, presented at this week's Friday studio. The habit you
build here is the one every later defense rests on.

*Provenance: fresh (robustness / sensitivity / placebo protocol) + course honest-analysis discipline + Purdue GenAI Studio reviewer bench | attack-the-analysis unit | alternative-specification grid, seeded honest-vs-searched demo, placebo and leave-one-out cells, and a multi-model reviewer-adjudication run (seed 464) | newly-constructed-from-source-concept*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Tell a **robustness check** from a **sensitivity check**, and name the four
   things you can vary to stress a claim: the **sample**, the **measurement**, the
   **specification**, and the **metric**.
2. Draw the line between honest robustness and **specification searching**, and
   state the one discipline that keeps you on the right side of it.
3. Run a **placebo test** and a **leave-one-out influence** check on real seeded
   code, and read what each one does and does not establish.
4. Compare what a human reviewer, a single AI, and a **multi-model** AI panel
   catch in the same analysis, and verify every flag against the data before you
   act on it.
5. Name the failure where two AI reviewers are wrong the same way, and say why
   their agreement is not confirmation.
6. Apply a pre-listed robustness plan and a verified AI-review trail to your own
   project's M9 research audit.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx        # for the attack-map diagram; if missing locally: pip install networkx (pre-installed in Colab)
from scipy import stats      # for p-values in the honest-vs-searched demo (pre-installed in Colab)

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

### 🤝 AI Research Partner

This week you point **Gemini** and the **Purdue GenAI Studio** reviewer bench at
your own analysis and ask them to break it. That is a good use. You commit your
own read first, then let a tool propose checks you might have missed, red-team a
sentence you wrote, or flag a weakness for you to verify against the data. A bad
use is letting a reviewer decide which flaw is real, or letting a fluent critique
stand in for a check you never ran.

**Never delegate these:** which robustness checks you commit to before you look,
which flagged issues the data actually confirms, and the final claim you defend.
A reviewer, human or AI, **proposes**. You verify, and the evidence decides. The
full never-delegate list lives in
[`ai_resources/human_responsibility_checklist.md`](https://github.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/ai_resources/human_responsibility_checklist.md).

**Commit first, then ask.** Every AI block below has you write your own answer
before you open a tool. Commit your work to git before a long AI session too, so
you can always compare what you wrote against what the tool changed.

> **A question that often comes up here:** *"If I am supposed to attack my own
> result, why not just let the AI do the attacking and save time?"* Because the
> attack is only worth something once you verify it. An AI reviewer will hand you
> a confident list of flaws, some real and some invented. The skill this week
> builds is telling those apart against your own data, and that skill is yours to
> keep, not to delegate.

---

# Lecture 1

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

A team runs its headline result one way and it is clearly significant. A reviewer
asks them to run it a slightly different but equally reasonable way. The result
shrinks to nothing.

Here is the question on the table: **is this result fragile, or did the team just
find the one way of slicing the data that happened to work?** Commit to an answer
before we go further. Then defend it. What single piece of information would let
you tell those two stories apart, a real-but-delicate finding versus a lucky
specification someone went looking for? Name it precisely. Getting that one thing
into the open is the whole job of Lecture 1.

## 1. Robustness, Sensitivity, and the Result That Lives or Dies by One Choice

**Guiding question:** *if your result only holds one way of slicing the data, is
it a finding, or an accident you went looking for?*

> *"Everyone shows me the version of the analysis that worked. I assume that one
> exists. My question is the next one. What happened to all the reasonable versions
> you could have run instead, and did you look at them before or after you saw this
> answer?"*
> — a **journal reviewer**, opening the first serious read your paper will get

Before you defend a number, you attack it. Two words name the two attacks, and
people mix them up constantly.

- **Robustness check**: you re-run the same finding under a different but equally
  defensible analytic choice, and see whether the answer stays. Example: you
  reported an average, so you also compute it after dropping the most extreme
  responses, and check that the story holds.
- **Sensitivity check**: you deliberately change an assumption to see how much the
  answer moves in response. Example: you assume the people who did not answer
  resemble those who did, then ask how far off your estimate would be if that
  assumption were wrong. Some fields call this a **sensitivity analysis**.

The two overlap in practice, and you do not need a bright line between them. What
you need is the instinct behind both: **a number you have not tried to break is a
number you cannot yet defend.**

In [ ]:
# Load the survey and build the headline estimate we will spend the week attacking.
lapop = load_course_data("lapop_brazil.csv")
assert lapop.shape == (10000, 10), "unexpected shape — flag this!"
print("✓ Loaded lapop_brazil.csv —", lapop.shape[0], "rows,", lapop.shape[1], "columns")

# HEADLINE ANALYSIS (stands in for YOUR OWN M8 estimate — swap yours in where marked):
# do people who trust the police more also report more support for the political system?
y = lapop["support_political_system"].astype(float)
sd = y.std(ddof=1)
hi = y[lapop["trust_police"] >= 5]      # "high" police trust
lo = y[lapop["trust_police"] <= 3]      # "low" police trust
headline_gap = (hi.mean() - lo.mean()) / sd
r = lapop["trust_police"].corr(lapop["support_political_system"])

print(f"  correlation(trust_police, support_political_system) = {r:.2f}")
print(f"  high-trust group n = {len(hi)},  low-trust group n = {len(lo)}")
print(f"  HEADLINE estimate: support gap = {headline_gap:.2f} SD (high minus low police trust)")

The **headline estimate** is one number that stands in for the whole finding.
Here it is the gap in average system-support between high-trust and low-trust
respondents, measured in standard-deviation units of support, so a gap of about
0.71 means the two groups sit roughly seven-tenths of a spread apart. That single
number is what you are about to attack.

You attack it from four directions. Any honest robustness plan varies one or more
of these, and the words matter because your M9 audit grades you on them.

- **Sample**: which rows you keep. Example: all respondents, or only those who
  gave a non-extreme ideology answer.
- **Measurement**: how you turn a concept into a number. Example: treating police
  trust as its full 1-to-7 scale, or splitting it at a high-versus-low cutoff.
- **Specification**: the bundle of modeling choices behind the estimate. Example:
  the raw gap, or the gap after removing whatever ideology explains.
- **Metric**: the yardstick you report. Example: a correlation, or a
  standardized gap between groups.

The picture below marks these four as the four handles a critic can grab.

In [ ]:
# The attack map: four handles a critic can turn before your estimate becomes a claim.
G = nx.DiGraph()
pos = {
    "Sample":        (0.0, 3.0),
    "Measurement":   (0.0, 2.0),
    "Specification": (0.0, 1.0),
    "Metric":        (0.0, 0.0),
    "Estimate":      (2.2, 1.5),
    "Claim":         (4.4, 1.5),
}
attack_edges = [("Sample", "Estimate"), ("Measurement", "Estimate"),
                ("Specification", "Estimate"), ("Metric", "Estimate")]
G.add_edges_from(attack_edges + [("Estimate", "Claim")])

fig, ax = plt.subplots(figsize=(10, 4.5))
handles = ["Sample", "Measurement", "Specification", "Metric"]
nx.draw_networkx_nodes(G, pos, nodelist=handles, node_color="#DCE6F1",
                       edgecolors="#4C72B0", linewidths=1.5, node_size=4200, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=["Estimate"], node_color="#DCE6F1",
                       edgecolors="#4C72B0", linewidths=2.5, node_size=5200, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=["Claim"], node_color="#E8E8E8",
                       edgecolors="#555555", linewidths=1.5, node_size=5200, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
nx.draw_networkx_edges(G, pos, arrowstyle="-|>", arrowsize=16, edge_color="#888888",
                       width=1.4, node_size=5000, ax=ax)
ax.set_title("The attack map: four handles you can turn to stress an estimate")
ax.axis("off")
plt.tight_layout()
plt.show()

print("✓ Attack map drawn.")
print("  Turn any handle a defensible other way and re-read the estimate.")
print("  If the estimate holds across every turn, it is robust. If it lives on one")
print("  setting of one handle, it is fragile, and the claim must say so.")

*In plain terms, the diagram says an estimate is not one number but a number
that depends on four choices, and a robustness check is you turning those choices
and watching what happens before the estimate hardens into a claim.*

> **A question that often comes up here:** *"If I try lots of versions, am I not
> just fishing for the answer I want?"* That is exactly the danger, and it is the
> heart of today's puzzle. The difference is not how many versions you run. It is
> **when you decide which ones count.** Turning every handle and reporting the
> whole picture is robustness. Turning handles until one gives you a star, then
> reporting only that one, is fishing. Section 2 gives that trap its real name.

### 🔮 Pause & Predict

You are about to re-run the headline gap across a grid of eight pre-listed
specifications: two samples, times two measurement cutoffs, times two ways of
handling ideology. The headline came out at about 0.71 SD. Before running the next
cell, commit: will the eight estimates all point the same direction, or will some
flip? And roughly how wide a range do you expect, low to high? Write one sentence
of reasoning. No peeking at the output.

### YOUR ANSWER HERE:

**Same direction or some flip:**

**My predicted range for the eight estimates (low to high), and why:**

---

### 🛠️ Run the Study: rerun the estimate across a pre-listed grid

Run the cell below. It defines the eight specifications **first**, then computes
the headline gap under each and plots the whole set as a **specification curve**,
a picture of one estimate under many defensible choices. Read the spread before
the next markdown cell. To make it yours later, replace the `lapop` headline with
your own M8 estimate where the comment marks it.

**🔴 Live in class: we run this one together.**
**Before you ask:** in one line, write the single specification you would most fear
was left off this grid, the one a hostile reviewer would add.

> 💡 **Gemini Prompt:** "Here is my pre-listed robustness grid for one estimate: I
> vary the sample (all rows / drop ideology extremes), the measurement cutoff
> (police trust >=5 vs <=3, or >=6 vs <=2), and whether I adjust for ideology.
> Act as a hostile peer reviewer. Name up to three additional, equally defensible
> specifications I did NOT list that could move this estimate, and for each say
> which direction you would expect it to push. Then tell me which one of your three
> you are least sure about, and why."
>
> **After running, verify (counters *illusion of completeness*: a long, tidy grid
> can still omit the one specification that would break the claim):**
> - [ ] Check each specification Gemini proposes against the four handles in the
>   attack map. If it names one you can actually run, add it and re-run. A
>   suggestion you cannot operationalize is not yet a check.
> - [ ] Confirm any number it quotes matches your printed curve, not a value it
>   guessed. The eight gaps your cell prints are the evidence, not its summary.
> - [ ] Decide yourself whether each proposed specification is defensible or just
>   different. Only you know which choices your question actually licenses.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Pre-list the checks FIRST, then run them. This ordering is the whole discipline.
def std_gap(df, cut_hi=5, cut_lo=3, adjust=False):
    """Headline gap: standardized support difference, high vs low police trust."""
    yy = df["support_political_system"].astype(float)
    if adjust:  # remove the linear ideology component from support, then compare
        xx = df["ideology"].astype(float)
        b = np.cov(xx, yy, ddof=1)[0, 1] / np.var(xx, ddof=1)
        yy = yy - b * (xx - xx.mean())
    g_hi = yy[df["trust_police"] >= cut_hi]
    g_lo = yy[df["trust_police"] <= cut_lo]
    return (g_hi.mean() - g_lo.mean()) / yy.std(ddof=1)

all_df = lapop
mid_df = lapop[(lapop["ideology"] >= 2) & (lapop["ideology"] <= 9)]  # drop ideology extremes
grid = []
for s_name, df in [("all rows", all_df), ("drop ideology extremes", mid_df)]:
    for c_name, (chi, clo) in [(">=5 vs <=3", (5, 3)), (">=6 vs <=2", (6, 2))]:
        for a_name, adj in [("raw", False), ("ideology-adj", True)]:
            grid.append((f"{s_name} | {c_name} | {a_name}", std_gap(df, chi, clo, adj)))

spec = pd.DataFrame(grid, columns=["specification", "gap_SD"])
print(spec.round(3).to_string(index=False))
print(f"\n  lowest  gap = {spec.gap_SD.min():.2f} SD")
print(f"  highest gap = {spec.gap_SD.max():.2f} SD")
print(f"  all eight point the same way (positive)? {bool((spec.gap_SD > 0).all())}")

# The specification curve: every pre-listed estimate, sorted, against zero.
order = spec.sort_values("gap_SD").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(range(len(order)), order.gap_SD, s=80, color="#4C72B0", zorder=3,
           label="one pre-listed specification")
ax.axhline(0, color="#555555", linestyle="-", linewidth=1)
ax.axhline(headline_gap, color="#C44E52", linestyle="--", linewidth=2,
           label=f"headline = {headline_gap:.2f} SD")
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"S{i+1}" for i in range(len(order))])
ax.set_xlabel("Specification (sorted smallest to largest)")
ax.set_ylabel("Support gap (SD units)")
ax.set_ylim(-0.1, 1.0)
ax.set_title("Specification curve: the headline gap under all eight pre-listed choices")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the direction is stable across every pre-listed specification.
assert (spec.gap_SD > 0).all(), "every pre-listed spec should point the same way"
assert 0.55 < spec.gap_SD.min() and spec.gap_SD.max() < 0.95, "spread drifted — is the data intact?"
print(f"✓ Self-check passed: all eight gaps are positive, ranging "
      f"{spec.gap_SD.min():.2f} to {spec.gap_SD.max():.2f} SD.\n"
      f"  The direction is robust; the magnitude depends on the choice.")

**Reading the curve.** Every one of the eight pre-listed specifications comes
back positive: high-trust respondents report more system-support than low-trust
respondents, no matter which defensible choice you make. The **direction is
robust.** The **magnitude** is not fixed. It ranges from about 0.63 SD at the
gentlest setting to about 0.90 SD at the widest cutoff. So the honest headline is
not a single number. It is a direction plus a range. Reporting only the 0.90 would
be picking the flattering corner. Reporting the whole curve is the robustness.

### 🔍 Reading the Evidence: robust, or cherry-picked?

You just saw the estimate survive every pre-listed choice. In the cell below,
write three things. First: the honest one-sentence headline this curve supports,
worded to carry the range, not just the best number. Second: name the compass
kind this claim is (descriptive or causal) and one thing the curve does **not**
establish. Third: suppose instead the curve had shown seven near-zero estimates
and one big positive one. What would you have to know about the order you ran them
in to tell fragile-but-real from cherry-picked?

### YOUR ANSWER HERE:

**My honest headline (direction + range):**

**Compass kind, and what the curve does NOT establish:**

**Seven-near-zero-and-one-big: what I would need to know:**

---

## 2. Robustness Versus Specification Searching

**Guiding question:** *what one discipline separates honest robustness from
fishing, when the two can produce the very same table?*

The puzzle's answer is not a statistic. It is a habit. Name the two things it
separates.

- **Specification searching**: running many analyses and reporting only the one
  that gave the result you wanted, without disclosing the search. Its everyday
  name is **p-hacking**, because trying enough versions eventually pushes some
  p-value below the usual 0.05 line by luck alone. Example: quietly testing ten
  subgroups and publishing the one where the effect was significant.
- **Pre-listed checks**: writing down the full set of robustness checks you will
  run, and committing to report all of them, **before** you look at any of their
  results. Example: listing the eight specifications above in your notebook, then
  running them and showing the whole curve.

The discipline is simply this: **pre-list, then run, then report everything.**
Same code, same table, opposite integrity, and the only difference is the order of
looking and deciding. Your M9 audit asks for your pre-listed checks precisely so
this order is on the record.

> **A question that often comes up here:** *"What if I run my pre-listed checks and
> then genuinely notice something new worth testing?"* You are allowed to notice.
> You are not allowed to relabel it. A check you thought of after seeing the data
> is an **exploratory** finding, a hypothesis for a future study, not a confirmed
> result. Write it down as exactly that. The honest move is to keep the two piles
> separate, never to quietly move a new idea into the pre-listed pile.

---

### ⏸ In class through here

**You have reached the end of Lecture 1's in-class block.** Everything below
deepens the same ideas and feeds your M9 audit. Finish it before the next lecture:
the honest-versus-searched demo that shows p-hacking manufacturing a result from
pure noise, the placebo test, the leave-one-out influence check, the design
choice, and the practice drill. Come back with your own pre-listed robustness plan
started.

---

### 🛠️ Hands-On: watch specification searching manufacture a result

Run the cell below. It builds a dataset where a "program" label is assigned by
coin flip and an outcome that has **nothing** to do with it. There is no real
effect to find. A **searcher** then slices the data into twenty arbitrary
subgroups and reports the most significant one. An **honest** analyst reports the
whole-sample result they pre-listed. Watch what each one concludes from identical,
meaningless data.

**🏠 Homework depth: run this on your own before the next lecture.**
**Before you ask:** in one sentence, predict how many of the twenty subgroups will
look "significant" (p < 0.05) by chance alone. Commit a number first.

> 💡 **Gemini Prompt:** "Here is a cell that assigns a program label at random,
> generates an unrelated outcome, then tests the program-outcome gap inside twenty
> arbitrary subgroups and reports the smallest p-value: [paste the next cell].
> First, explain why testing twenty independent null comparisons at the 0.05 level
> is expected to produce about one 'significant' result even when nothing is going
> on. Then give me a rule I could pre-commit to that would stop me from reporting
> that lucky subgroup as a finding."
>
> **After running, verify (counters *plausible-but-wrong-method*: a fluent story
> about multiple comparisons can still misstate what your cell actually found):**
> - [ ] Confirm the count of 'significant' subgroups Gemini discusses matches the
>   number your cell printed, not a round figure it assumed.
> - [ ] Check that the honest whole-sample p-value your cell reports is genuinely
>   non-significant. If Gemini implies the data hold a real effect, it misread the
>   setup. There is none by construction.
> - [ ] Confirm the rule it proposes is one you could actually pre-list (for
>   example, 'report every subgroup, or none'), not a vague 'be careful'.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Honest vs searched, on data with NO real effect by construction.
rng_ss = np.random.default_rng(SEED)   # fresh seeded generator, reproducible on its own
n = 2000
program = rng_ss.integers(0, 2, n)           # a coin-flip label: no effect on anything
outcome = rng_ss.normal(0, 1, n)             # pure noise, unrelated to program
subgroup = rng_ss.integers(0, 20, n)         # 20 arbitrary, meaningless slices

# THE SEARCHER: test the program->outcome gap inside every subgroup, keep the best p.
searched = []
for s in range(20):
    m = subgroup == s
    a, b = outcome[m & (program == 1)], outcome[m & (program == 0)]
    gap = a.mean() - b.mean()
    p = stats.ttest_ind(a, b).pvalue
    searched.append({"subgroup": s, "gap": gap, "p_value": p})
searched = pd.DataFrame(searched).sort_values("p_value").reset_index(drop=True)
n_sig = int((searched.p_value < 0.05).sum())
best = searched.iloc[0]

# THE HONEST ANALYST: the pre-listed whole-sample test, reported no matter what.
honest_gap = outcome[program == 1].mean() - outcome[program == 0].mean()
honest_p = stats.ttest_ind(outcome[program == 1], outcome[program == 0]).pvalue

print(f"SEARCHER's headline: subgroup {int(best.subgroup)}, gap = {best.gap:+.3f}, "
      f"p = {best.p_value:.4f}  (looks significant!)")
print(f"  significant subgroups found (p < 0.05): {n_sig} of 20")
print(f"HONEST whole-sample result: gap = {honest_gap:+.3f}, p = {honest_p:.3f}  (correctly null)")
print("\n  Same data. The searcher 'found' an effect; the honest analyst found none.")

# Picture it: 20 subgroup gaps, the 'significant' ones flagged, against the true zero.
fig, ax = plt.subplots(figsize=(9, 5))
sig = searched.p_value < 0.05
ax.scatter(searched.subgroup[~sig], searched.gap[~sig], s=70, color="#4C72B0",
           zorder=3, label="subgroup gap (not significant)")
ax.scatter(searched.subgroup[sig], searched.gap[sig], s=140, color="#DD8452",
           marker="X", zorder=4, label="subgroup the searcher would report (p < 0.05)")
ax.axhline(0, color="#555555", linestyle="--", linewidth=2, label="the truth: no effect")
ax.set_xlabel("Arbitrary subgroup")
ax.set_ylabel("Program-outcome gap")
ax.set_title("Specification searching: 'significant' subgroups appear in pure noise")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: the data are null, yet the search still 'finds' something.
assert honest_p > 0.05, "the honest whole-sample test should be non-significant"
assert best.p_value < 0.05, "the searcher should surface at least one 'significant' slice"
print(f"✓ Self-check passed: the honest test is null (p = {honest_p:.2f}), but the "
      f"searcher\n  reported subgroup {int(best.subgroup)} at p = {best.p_value:.4f}. "
      f"The star is an artifact of looking {len(searched)} times.")

**Reading the evidence.** The data were built with no real effect at all. Yet the
searcher, by slicing twenty ways and reporting only the smallest p-value, produced
a headline that reads as strongly significant. That is not fraud. It is what
happens when you decide which analysis counts *after* you see the results. The
honest whole-sample test, pre-listed and reported no matter what, correctly found
nothing. The lesson is blunt. **A significant result you went looking for is not
the same kind of thing as one that survived a test you committed to in advance.**

> **A question that often comes up here:** *"So is running many specifications
> always wrong?"* No. Running many is exactly what a specification curve does, and
> that is good practice. The wrong part is running many and *showing one*. Report
> all twenty subgroups and the picture is honest, because the reader sees the
> nineteen nulls next to the one star and judges accordingly. Hide the nineteen and
> the same star becomes a lie of omission.

## 3. Placebo Tests and Influence: Two More Ways to Break a Result

**Guiding question:** *how do you check that your result is not an artifact of your
procedure, or the work of a single unusual data point?*

Two more attacks belong in every audit, and they answer different worries.

- **Placebo test**: you run your exact analysis on a version where the effect
  *cannot* exist, and check that it comes back near zero. Example: replace your
  real treatment groups with a coin-flip label, and confirm the "effect" vanishes.
- **Falsification test**: you check a prediction that *should* be false if your
  story is right, and confirm it is false. Example: your program started in March,
  so you check for an "effect" in February, before it existed.
- **Negative test**: the same idea aimed at an outcome your cause could not touch.
  Also called a **negative control**. Example: a tutoring program should not change
  a participant's shoe size, so a shoe-size "effect" would signal a broken analysis.

All three share one move: point the machinery at a place where the answer must be
zero, and make sure it says zero. If it does not, the machinery, not the world, is
producing your result.

### 🛠️ Hands-On: a placebo test on the headline estimate

Run the cell. It keeps the real system-support answers but throws away the real
police-trust grouping, replacing it with a coin-flip "high/low" label of the same
sizes. Then it runs the exact standardized-gap machinery from Section 1. If our
0.71 SD headline were an artifact of the procedure, a fake grouping would still
produce a gap. Watch whether it does.

**🏠 Homework depth: run this on your own.**
**Before you ask:** commit a number. What gap (in SD) do you expect the fake
grouping to produce, and why?

> 💡 **Gemini Prompt:** "For my own analysis, I want three placebo or falsification
> tests, not a re-run of the main result. My design is [describe your groups,
> outcome, and timing in two sentences]. Propose three checks where the estimate
> *should* come back near zero if my analysis is sound: one placebo label, one
> pre-period or falsification outcome, and one negative-control outcome my cause
> could not plausibly affect. For each, state the exact zero I should expect and
> what a non-zero result would tell me is broken."
>
> **After running, verify (counters *confident fabrication*: a plausible-sounding
> placebo can be one your data cannot actually run):**
> - [ ] For each proposed test, confirm the variable it needs exists in your data.
>   A placebo you cannot compute is a suggestion, not a check.
> - [ ] Compare the fake-grouping gap your cell printed against the near-zero you
>   predicted. If the two disagree, trust the printout and re-read the code.
> - [ ] Reject any 'negative control' that your cause plausibly *could* affect. A
>   bad negative control fails silently, and only you know the mechanism.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Placebo test: replace the real trust grouping with a coin flip, keep the outcome.
rng_pl = np.random.default_rng(SEED)          # reproducible on its own
y_all = lapop["support_political_system"].astype(float).values
sd_all = y_all.std(ddof=1)
n_group = int((lapop["trust_police"] >= 5).sum()) + int((lapop["trust_police"] <= 3).sum())

def placebo_gap():
    idx = rng_pl.choice(len(y_all), size=n_group, replace=False)  # a random slice
    label = rng_pl.integers(0, 2, n_group)                        # a meaningless hi/lo
    vals = y_all[idx]
    return (vals[label == 1].mean() - vals[label == 0].mean()) / sd_all

one_placebo = placebo_gap()
placebos = np.array([placebo_gap() for _ in range(500)])

print(f"Real headline gap (Section 1)      : {headline_gap:.3f} SD")
print(f"One placebo (fake grouping) gap    : {one_placebo:+.3f} SD")
print(f"500 placebo gaps: mean = {placebos.mean():+.3f} SD, sd = {placebos.std():.3f} SD")
print(f"The real gap sits about {headline_gap / placebos.std():.0f} placebo-SDs from zero.")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(placebos, bins=25, color="#B0B0B0", edgecolor="white",
        label="500 placebo gaps (fake groupings)")
ax.axvline(headline_gap, color="#4C72B0", linestyle="-", linewidth=2.5,
           label=f"real headline = {headline_gap:.2f} SD")
ax.axvline(0, color="#555555", linestyle="--", linewidth=1.5, label="zero (the placebo target)")
ax.set_xlabel("Standardized support gap (SD units)")
ax.set_ylabel("Number of placebo runs")
ax.set_xlim(-0.15, 0.85)
ax.set_title("Placebo test: fake groupings cluster at zero; the real result does not")
ax.legend(loc="upper center")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: placebo groupings return ~0; the real result is nowhere near them.
assert abs(placebos.mean()) < 0.05, "placebo gaps should center on zero"
assert headline_gap > placebos.max() + 0.3, "the real gap should sit far above every placebo"
print(f"✓ Self-check passed: 500 fake groupings averaged {placebos.mean():+.3f} SD "
      f"(essentially zero),\n  while the real {headline_gap:.2f} SD gap sits far outside "
      f"the placebo cloud. The result is\n  not an artifact of the procedure.")

**Reading the evidence.** The placebo passed. Fake groupings, built to have no
real basis, cluster tightly at zero, while the real 0.71 SD gap sits far outside
that cloud. This does not prove the relationship is causal. It rules out one
specific worry: that the machinery itself manufactures a gap from nothing. A
placebo test cannot rescue a biased sample or turn a correlation into a cause. It
answers exactly one question, and it answered it well here.

### 🛠️ Hands-On: leave-one-out influence

One unusual point can carry an entire result on its back. Two terms name the risk.

- **Outlier**: a data point far from the rest. Example: in a class where everyone
  reports studying one to three hours, one respondent who reports forty.
- **Influential point**: a point whose *removal* changes your conclusion. Not every
  outlier is influential, and not every influential point looks extreme, so you
  test for influence directly. The tool is **leave-one-out**: drop each point in
  turn, recompute, and see which single drop moves the answer most.

Run the cell. It builds a small dataset with one planted high-leverage point, then
leaves out each point in turn to find the one whose removal flips the correlation.

In [ ]:
# A small dataset: no real relationship, plus ONE high-leverage point.
rng_inf = np.random.default_rng(SEED)
x = rng_inf.normal(0, 1, 30)
y_inf = rng_inf.normal(0, 1, 30)
x = np.append(x, 6.0)          # the planted point sits far from the cloud, on both axes
y_inf = np.append(y_inf, 6.0)

full_corr = np.corrcoef(x, y_inf)[0, 1]
loo = np.array([np.corrcoef(np.delete(x, i), np.delete(y_inf, i))[0, 1]
                for i in range(len(x))])   # leave-one-out correlations
most_influential = int(np.argmax(np.abs(full_corr - loo)))

print(f"Correlation using all {len(x)} points : {full_corr:+.3f}  (looks like a real link)")
print(f"Most influential point index          : {most_influential}  "
      f"at (x, y) = ({x[most_influential]:.1f}, {y_inf[most_influential]:.1f})")
print(f"Correlation with that one point removed: {loo[most_influential]:+.3f}  (the link vanishes)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(x[:-1], y_inf[:-1], s=55, color="#4C72B0", label="ordinary points")
axes[0].scatter(x[-1], y_inf[-1], s=160, color="#DD8452", marker="X",
                label="the one influential point")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title(f"Full-sample correlation = {full_corr:+.2f}")
axes[0].legend(loc="upper left")
axes[1].scatter(range(len(loo)), loo, s=45, color="#4C72B0")
axes[1].scatter(most_influential, loo[most_influential], s=160, color="#DD8452", marker="X")
axes[1].axhline(0, color="#555555", linestyle="--", linewidth=1.5)
axes[1].set_xlabel("point left out")
axes[1].set_ylabel("correlation without that point")
axes[1].set_title("Leave-one-out: one drop moves the answer")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: one point is doing all the work.
assert full_corr > 0.4, "the full-sample correlation should look substantial"
assert abs(loo[most_influential]) < 0.1, "removing the influential point should collapse it"
print(f"✓ Self-check passed: the full correlation of {full_corr:.2f} collapses to "
      f"{loo[most_influential]:+.2f}\n  when one point is removed. A result that rests on a "
      f"single observation is not yet robust.")

### ⚖️ Make a Design Choice: which three checks do you pre-list?

*(Human-first: settle your own plan and its defense before you ask any AI.)*

Your M9 audit asks for a short, pre-listed robustness plan for your own estimate.
You cannot run everything. Pick the **three** checks you will commit to, from the
menu below, and defend the set. A good plan spends its three checks on the choices
a hostile reviewer is most likely to attack, not the ones easiest to run.

- **A.** Rerun across two alternative **samples** (for example, with and without a
  subgroup you suspect is unusual).
- **B.** Rerun under two alternative **measurements** of your key variable.
- **C.** A **placebo test** (a grouping or period where the effect cannot exist).
- **D.** A **leave-one-out influence** check for a single point carrying the result.
- **E.** A **specification curve** across several defensible modeling choices.

In the cell below, name your three, in order, and defend the set: which reviewer
attack does each one answer, and why did you leave the other two off?

### YOUR ANSWER HERE:

**My three pre-listed checks (in order):**

**The reviewer attack each one answers:**

**Why I left the other two off:**

---

### 📝 Practice: robustness check, or specification search?

*(Human-first: classify all six yourself before any AI. The sorting is the skill
being graded.)*

For each analysis below, label it a **robustness check** (honest) or a
**specification search** (fishing), and in a few words say what makes it one or the
other. Answer in the cell that follows.

- **A.** You list five subgroup analyses in your notebook, run all five, and report
  all five, including the three that showed nothing.
- **B.** You try eleven outcome variables, find one that reaches p < 0.05, and put
  only that one on your poster.
- **C.** You rerun your main estimate dropping the top and bottom 1% of responses,
  and report that it barely moved.
- **D.** You test your effect in a pre-treatment period where it cannot exist, and
  report that it came back near zero.
- **E.** After seeing a null result, you keep re-defining "treated" until the effect
  turns significant, then report that definition as if it were your first.
- **F.** For the influence dataset above, you report the correlation as +0.53
  without mentioning that removing one point sends it to about zero.

### YOUR ANSWER HERE:

**A:**   **B:**   **C:**

**D:**   **E:**   **F:**

---

You have four attacks in hand now: the specification grid, the honest-versus-
searched contrast, the placebo test, and the influence check. Lecture 2 turns the
attackers loose. You will hand your analysis to a human reviewer, to one AI, and to
a panel of AI models, and learn to tell a real flaw from a confident invention.

---

---

# Lecture 2

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

You paste your analysis into three different AI reviewers and ask each for the
single most serious flaw. Reviewer 1 says the result is driven by one region.
Reviewer 2 says your cutoff is doing all the work. Reviewer 3 says you are talking
about a correlation as if it were a cause. **All three are confident. All three
disagree.**

Here is the question on the table: **which one is real, and how would you tell
without simply trusting the most confident-sounding one?** Commit an answer before
we go further. Then defend it. For each of the three flaws, name the one check you
could run against your own data that would either confirm it or kill it. If a flaw
has no such check, that tells you something important about it. Sorting the three
is the whole job of Lecture 2.

## 4. Human, One AI, and a Panel of AIs Read Your Analysis

**Guiding question:** *when a human, one model, and a panel of models read the same
analysis, who catches what, and would you have caught it yourself?*

> *"I do not want the AI's verdict on your work. I want to know which of the things
> it flagged you actually checked, which turned out to be real, and which it made
> up. Show me that, and I will trust your judgment. Show me its transcript, and I
> will trust neither."*
> — a **thesis advisor**, on the difference between using a reviewer and hiding
> behind one

You have three kinds of reviewer available, and they fail differently.

- **Human reviewer feedback**: a peer reads your work. Slow, but grounded in real
  understanding and hard to fool with fluent nonsense. Example: a classmate who
  says "your Table 1 does not match your headline number."
- **Single-AI review**: one model critiques your analysis. Fast and wide-ranging,
  but confidently wrong in patterned ways. Example: Gemini lists eight possible
  flaws in seconds, one of which does not exist.
- **Multi-model comparison**: you run the same analysis past several different
  models and compare. Where they **agree**, you get a candidate to check. Where
  they **disagree**, you learn which issues are contested. And you watch for
  **confident fabrication**: a flaw a reviewer invents and states with full
  certainty even though the data refute it. Example: a model insists your result is
  driven by one region, with no hedging, when dropping that region barely moves the
  gap. This is the failure named in the AI error taxonomy that you will point to by
  name later in this lecture, and the only cure is running the check, never weighing
  how sure the reviewer sounded.

One more failure has its own name, because it is the trap of using many models.

- **Correlated errors**: two models (or two roles of one model) make the *same*
  mistake, so their agreement feels like confirmation when it is just one flaw
  echoed twice. Example: two models both assume your data are a random sample and
  both praise a generalization your frame cannot support. Agreement only reassures
  when the checks are independent.

The move that makes all of this safe is the same one all week: **verify every
flagged issue against the data.** A flaw is real when a check confirms it, not when
a reviewer sounds sure.

### 🔮 Pause & Predict

The next cell takes the three reviewer flags from the puzzle and runs each one
against the real data: it drops the largest region and rechecks the gap, it
recomputes the relationship without the high-versus-low cutoff, and it examines the
causal-language flag. Before running, commit: which of the three do you predict the
data will **confirm**, and which will it **refute**? Write one sentence naming the
flaw you think survives, and why.

### YOUR ANSWER HERE:

**Which flag(s) the data will confirm, which it will refute:**

**The flaw I think survives, and why:**

---

### 🛠️ Hands-On: adjudicate three reviewers against the data

Run the cell. It encodes the three reviewers' flags on our headline result, then
runs the data check that confirms or kills each, and prints a verdict for every
one. Read the verdicts before the next markdown cell.

**🔴 Live in class: we run this one together.**
**Before you ask:** write one sentence naming which flag you expect to be the real
one, so you have a committed answer to compare against the tool.

> 💡 **Gemini Prompt:** "Act as a hostile peer reviewer, not a cheerleader. Here is
> a one-paragraph summary of my analysis and its headline claim: [paste your own
> summary]. Name the single most serious flaw you can find, and state exactly what
> data check would confirm or refute it. Do not rewrite my analysis and do not
> list more than one flaw. After you answer, argue against your own objection: give
> me the strongest reason a fair reader would say the flaw is not fatal."
>
> **After running, verify (counters *sycophantic agreement*: a reviewer that
> praises your work has reviewed your ego, not your evidence):**
> - [ ] If the model calls your analysis 'sound' or offers only mild praise, push
>   back: "assume it is flawed and name the worst problem." Weak praise is a red flag.
> - [ ] Run the data check it names. The flaw is real only if your own output
>   confirms it. Match its claim against the numbers this cell printed.
> - [ ] Confirm its self-argument is genuine, not a throwaway. A reviewer that
>   cannot argue both sides has not understood the flaw.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Three AI reviewers, three different 'most serious flaws.' Check each on the data.
biggest_muni = lapop["municipality"].value_counts().idxmax()
gap_drop_region = std_gap(lapop[lapop["municipality"] != biggest_muni])   # Reviewer 1's check
cont_corr = lapop["trust_police"].corr(lapop["support_political_system"])   # Reviewer 2's check

reviews = [
    {"reviewer": "Model 1", "flag": "driven by one region; drop the biggest and it vanishes",
     "data_check": f"gap without largest region = {gap_drop_region:.2f} SD (vs {headline_gap:.2f})",
     "verdict": "REFUTED" if abs(gap_drop_region - headline_gap) < 0.1 else "CONFIRMED"},
    {"reviewer": "Model 2", "flag": "the >=5 / <=3 cutoff is manufacturing the gap",
     "data_check": f"continuous correlation = {cont_corr:.2f} (same direction, no cutoff used)",
     "verdict": "REFUTED" if cont_corr > 0.15 else "CONFIRMED"},
    {"reviewer": "Model 3", "flag": "a correlation reported as if police trust CAUSES support",
     "data_check": "no data check can settle this — it is a claim-boundary flaw, not a number",
     "verdict": "REAL (design flaw)"},
]
verdicts = pd.DataFrame(reviews)
print(verdicts.to_string(index=False))
print("\n  Two confident flags were refuted by the data. The one that survives is the")
print("  boundary flaw no robustness check can fix: an observational gap is not a cause.")

In [ ]:
# Self-check: the two data-checkable flags fail; the real flaw is the boundary one.
assert abs(gap_drop_region - headline_gap) < 0.1, "dropping the largest region barely moves the gap"
assert cont_corr > 0.15, "the continuous correlation should confirm the direction without any cutoff"
print(f"✓ Self-check passed: without the largest region the gap is {gap_drop_region:.2f} SD "
      f"(vs {headline_gap:.2f}),\n  and the cutoff-free correlation is {cont_corr:.2f}. "
      f"Both 'flaws' are refuted; the real one is\n  that this is a correlation, not a cause.")

### 🔍 Reading the Evidence: the loudest reviewer is not the truest

Three confident reviewers, three different verdicts, and the data sorted them out.
The two flags that *sounded* like methods critiques, one region and one cutoff,
both dissolved the moment a check touched them. The flag that survived was not
about the numbers at all. It was the boundary flaw: reporting an observational
correlation with the grammar of a cause. No robustness check can fix that, because
it is a claim problem, not a computation problem. In the cell below, write three
things. First: which reviewer was right, and the exact check (or reasoning) that
decided it. Second: what a **research audit** should record about the two refuted
flags, so a reader knows you checked them. Third: name the failure mode from the AI
error taxonomy that the two confident-but-wrong reviewers illustrate.

### YOUR ANSWER HERE:

**Which reviewer was right, and the check that decided it:**

**What my research audit records about the two refuted flags:**

**The AI-error-taxonomy failure the wrong reviewers illustrate:**

---

> **A question that often comes up here:** *"If a panel of AI models all agree on
> a flaw, is that enough to act on it?"* Not by itself. Agreement across models can
> be **correlated error**: they may share training data and a blind spot, so they
> repeat one another rather than confirm one another. Treat unanimous AI agreement
> as a strong candidate to check, never as the check itself. The independent
> verification is still a real check against your data or a human who reasons from
> scratch.

---

### ⏸ In class through here

**You have reached the end of Lecture 2's in-class block.** Everything below feeds
your M9 poster draft and research audit, due at Friday's studio. Finish it before
then: adapt the adversarial prompt to your own analysis, interrogate what the panel
returns, make the AI-free call on which flaws are real, and draft your audit spine.
The 🧑‍⚖️ Human-Only Checkpoint is the never-delegate core of the audit, so give it
real time.

---

### 🔁 Modify the Prompt

The live prompt above asked one AI for one flaw. Now build the **multi-model
comparison** yourself. You will run the same summary past two or three different
models (or GenAI Studio roles) and compare what they catch.

> **Base prompt (from above):** "Act as a hostile peer reviewer. Here is my analysis
> summary: [paste]. Name the single most serious flaw and the exact data check that
> would confirm or refute it."

Rewrite it so you run it in **three different tools or roles** and then ask a fourth
question you supply: *"Here are three flaws three reviewers named: [A], [B], [C].
Which two are most likely the same underlying concern in different words, and which
one is genuinely separate?"* In the cell below: paste your analysis summary, the
three flags you collected, and your prediction of which flag will survive a data
check. Then run each check and note what actually happened.

**🏠 Homework depth: run this on your own.**

> 💡 **Gemini Prompt:** "I collected three 'most serious flaw' verdicts on my
> analysis from three different reviewers: [paste A, B, C]. For each flaw, tell me
> the single data check that would confirm or refute it, in one line. Then tell me
> which of the three, if any, is a claim-boundary problem that NO data check can
> settle, and must be fixed by narrowing the claim instead."
>
> **After running, verify (counters *correlated errors*: three reviewers echoing
> one blind spot can feel like three confirmations):**
> - [ ] Run each data check it proposes against your own output before believing
>   any flag. A flaw with no runnable check is either a boundary problem or nothing.
> - [ ] Ask whether two of the three flags are the same concern reworded. If so,
>   you have two echoes, not two independent hits.
> - [ ] Confirm the 'no data check can settle this' flag really is a boundary
>   issue you must fix by narrowing the claim, not a check you were too lazy to run.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

### YOUR ANSWER HERE:

**My analysis summary:**

**The three flags I collected (and from which tools/roles):**

**My prediction of which flag survives a data check:**

**What each data check actually showed:**

---

### 🔬 Interrogate the Output

You now have a panel's worth of flags. Do not accept them, and do not dismiss them.
Interrogate the panel against the data, using four checks. Answer each in the cell
below.

- **Claims:** does each flag point to a real problem your output confirms? A flag
  that a data check refutes (like the two in Section 4) is confident fabrication,
  and you record it as refuted.
- **Assumptions:** what does a flag assume about your design? A reviewer that
  assumes you ran an experiment when you ran a survey is critiquing a study you did
  not do.
- **Missing information:** which real weakness did the whole panel *miss*? If none
  of them named your actual biggest limitation, that is the most important finding
  of the exercise.
- **Overstatements:** does any flag sound more certain than its evidence? Flag the
  exact words, especially a causal verb attached to your observational design.

And the rule that governs the whole week: **code that runs without error is not
code that is correct, and a reviewer that sounds certain is not a reviewer that is
right.** Verify the number, and verify the flaw.

### YOUR ANSWER HERE:

**Claims (which flags the data confirm vs refute):**

**Assumptions (any flag critiquing a study I did not run):**

**Missing information (the real weakness the whole panel missed):**

**Overstatements (exact words):**

---

### 🧑‍⚖️ Human-Only Checkpoint

Close every AI tool for this one. No Gemini, no GenAI Studio, no notebook search.
This is a never-delegate decision: **which claims your evidence justifies** is
yours to make and defend. In the cell below, write, in your own words:

1. **The one claim your analysis actually supports**, worded to its real boundary
   (the compass kind, and the range from your specification curve, not one number).
2. **The one flaw you now believe is real**, whether a reviewer named it or not,
   and the check or reasoning that convinced you.
3. **The decision you refuse to automate**: the judgment about your own project that
   you will not hand to any reviewer, human or AI, and why it has to stay yours.

If you cannot yet write the claim honestly, that is a finding, not a failure. It
tells you which attack you still have to answer before the studio.

### YOUR ANSWER HERE:

**1. The claim my analysis supports (boundary + range):**

**2. The one real flaw, and what convinced me:**

**3. The decision I refuse to automate, and why:**

---

### 🎯 Project Transfer: the spine of your M9 research audit

*(Human-first: draft every piece yourself. AI may critique it after, but the audit
you defend at the studio has to be reasoned by you.)*

This is where the week becomes your **M9 poster draft and research audit**. A
**research audit** is the graded record of how hard you tried to break your own
claim, and what survived. In the cell below, draft its spine for *your* project.

1. **Pre-listed checks:** the three robustness or placebo checks you committed to
   *before* looking, from your Design Choice above, each tied to the reviewer
   attack it answers.
2. **What survived:** your headline claim after the checks, worded to carry its
   range and its compass boundary, with a note on any check that moved it and by
   how much.
3. **The verified AI-review trail:** the real flaws your human and multi-model
   reviewers found, each marked confirmed or refuted *by a data check*, plus the
   most confident wrong flag you caught.
4. **The remaining limitation:** the one weakness no check fixed, stated as
   expertise, not hidden.

These four pieces are your research audit, and they are what you walk a skeptic
through at Friday's studio. Write them so a reader who has never seen your project
could tell exactly how far you pushed on your own number.

### YOUR ANSWER HERE:

**1. My pre-listed checks (each + the attack it answers):**

**2. What survived (headline + range + compass boundary):**

**3. Verified AI-review trail (real hits confirmed/refuted by a check + the loudest wrong flag):**

**4. The remaining limitation no check fixed:**

---

### 📒 AI Research Ledger

Log every AI use from this notebook. One worked row is filled in as a model, and it
models the discipline this week teaches: **the AI proposed a flaw; you checked it
against the data before believing it.** This notebook offers five prompts, one live
per lecture and three homework-depth, so your ledger carries a row for each one you
actually ran. The ledger is a graded habit, not paperwork. It is how your audit
proves the review was verified, not just run.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| Find the single most serious flaw in my headline analysis | Gemini | "Act as a hostile reviewer; name the worst flaw and the data check that settles it; do not rewrite." | Flagged 'driven by one region' as fatal, with full confidence | Rejected it: recomputed the gap without the largest region and it moved under 0.01 SD | Reran the leave-one-region check myself; the flag was refuted | A reviewer this confident about a wrong flaw may be right about a different one I have not checked | *(your name)* |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

---

### 🛡️ Exit Defense

Defense #09 — write, in your own words:

1. **The claim I can defend:** one bounded sentence, drawn from today's analysis or
   your own project, that survived your pre-listed checks.
2. **Its boundary:** what this evidence does NOT establish. Name the compass kind
   (descriptive or causal) and whether any robustness or placebo check moved it.
3. **My uncertainty and limitations:** one sentence naming the range from your
   specification curve, or the one flaw no check could fix.
4. **AI check:** what you delegated to a reviewer, whether you **accepted, changed,
   or rejected** what it returned, and the specific data check that decided it.

### YOUR ANSWER HERE:

**1. The claim I can defend:**

**2. Its boundary (compass kind + what moved it):**

**3. My uncertainty and limitations:**

**4. AI check (what I delegated, how I verified):**

---

## 5. Wrap-Up

Across two lectures you learned to attack your own analysis before anyone else
could. You told a robustness check from a sensitivity check, and reran one estimate
across alternative samples, measurements, specifications, and metrics, reading the
specification curve for where the claim held and where it moved. You watched
specification searching manufacture a significant result from pure noise, which is
why pre-listing your checks is the whole discipline. You ran a placebo test that
ruled out a procedural artifact, and a leave-one-out check that exposed a result
resting on a single point. Then you handed your analysis to a human, one AI, and a
panel, and learned that the loudest reviewer is not the truest. A flaw is real only
when the data confirm it.

> **"Pre-list your checks, run all of them, and report the whole picture. A result
> that survived an attack you committed to in advance is worth ten that survived
> only the version you chose to show. The audit is where you prove you tried to
> break your own claim."**

Next week the work turns outward to the poster: criticism, delivery, and the
public defense of a bounded claim. Your M9 poster draft and research audit are due
at Friday's studio, where the required GenAI Studio Poster Critic and Robustness
reviewers meet your draft. Bring your pre-listed checks, your surviving headline,
and your verified review trail. You are about to defend, out loud, the number you
spent this week trying to break.

---

## 6. Sources & Provenance

**Provenance of this notebook:**
- *fresh (robustness / red-team protocol) + course honest-analysis discipline | the attack-the-analysis toolkit: robustness vs sensitivity, placebo/falsification/negative tests, pre-listed checks vs specification searching, influence | newly-constructed-from-source-concept*
- *RDSS ch. 10 'Diagnosing designs' + ch. 11 'Redesigning' | diagnosing a design's properties and revising when a check reveals a weakness | companion reading (concept-level, honors non-quant audience)*
- *project/poster/redteam_review_protocol.md | the four-audit red-team vocabulary and the reviewer-proposes-you-verify rule | referenced (previews the M-level poster red-team)*
- *lapop_brazil.csv | rdss package data | a headline trust-and-support gap reused as the analysis under attack, then stress-tested with a seeded robustness grid, placebo test, and multi-model reviewer adjudication | adapted (10,000-row resample-with-replacement; teaching/planning only, not a population claim)*
- *fresh | the seeded honest-vs-searched (p-hacking) demo and the one-outlier leave-one-out dataset | newly-constructed-from-source-concept (seed 464)*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 10 'Diagnosing designs' and ch. 11 'Redesigning'. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).
- *(Optional)* Bergstrom, C. & West, J. *Calling Bullshit*, on results that vanish
  under reanalysis: [callingbullshit.org/tools](https://callingbullshit.org/tools/).

---

<center>

Thank you!

</center>